In [ ]:
# seismo-benchmark-control
import os as _seismo_bench_os
import sys as _seismo_bench_sys
import time as _seismo_bench_time
from pathlib import Path as _SeismoBenchPath

_seismo_bench_here = _SeismoBenchPath.cwd()
for _seismo_bench_root in [_seismo_bench_here, *_seismo_bench_here.parents]:
    if (
        (_seismo_bench_root / "benchmarking" / "seismo_bench.py").exists()
        or (_seismo_bench_root / "seismo_bench.py").exists()
    ):
        if str(_seismo_bench_root) not in _seismo_bench_sys.path:
            _seismo_bench_sys.path.insert(0, str(_seismo_bench_root))
        break

try:
    from benchmarking.seismo_bench import make_context as _seismo_bench_make_context
    from benchmarking.seismo_bench import start_run as _seismo_bench_start_run
except ModuleNotFoundError:
    from seismo_bench import make_context as _seismo_bench_make_context
    from seismo_bench import start_run as _seismo_bench_start_run

def _seismo_bench_float_from_env(name):
    value = _seismo_bench_os.environ.get(name, "").strip()
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        return None

_SEISMO_BENCH_CONTEXT = _seismo_bench_make_context(
    notebook="Noise/Probabilistic Power Spectral Densities.ipynb",
    environment=_seismo_bench_os.environ.get("SEISMO_BENCH_ENV", "unset"),
    browser=_seismo_bench_os.environ.get("SEISMO_BENCH_BROWSER", "unset"),
    device=_seismo_bench_os.environ.get("SEISMO_BENCH_DEVICE", "unset"),
    run_index=int(_seismo_bench_os.environ.get("SEISMO_BENCH_RUN_INDEX", "0")),
    platform_peak_memory_mb=_seismo_bench_float_from_env("SEISMO_BENCH_PLATFORM_PEAK_MEMORY_MB"),
    notes=_seismo_bench_os.environ.get("SEISMO_BENCH_NOTES", ""),
)
_SEISMO_BENCH_RUN = _seismo_bench_start_run(_SEISMO_BENCH_CONTEXT)
_SEISMO_BENCH_TOTAL_T0 = _seismo_bench_time.perf_counter()
_SEISMO_BENCH_PENDING_PHASE = None
_SEISMO_BENCH_PENDING_T0 = None

def _seismo_bench_start_phase(phase):
    global _SEISMO_BENCH_PENDING_PHASE, _SEISMO_BENCH_PENDING_T0
    _SEISMO_BENCH_PENDING_PHASE = phase
    _SEISMO_BENCH_PENDING_T0 = _seismo_bench_time.perf_counter()

def _seismo_bench_save_silently():
    try:
        _SEISMO_BENCH_RUN.save()
    except Exception as exc:
        print(f"Benchmark save failed: {exc}")

def _seismo_bench_post_run_cell(result):
    global _SEISMO_BENCH_PENDING_PHASE, _SEISMO_BENCH_PENDING_T0
    if _SEISMO_BENCH_PENDING_PHASE is None:
        return
    if _SEISMO_BENCH_PENDING_T0 is None:
        _SEISMO_BENCH_PENDING_PHASE = None
        return
    raw_cell = getattr(getattr(result, "info", None), "raw_cell", "")
    if "seismo-benchmark-control" in raw_cell:
        return
    elapsed = _seismo_bench_time.perf_counter() - _SEISMO_BENCH_PENDING_T0
    error = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
    success = error is None and bool(getattr(result, "success", True))
    notes = ""
    if error is not None:
        notes = f"{type(error).__name__}: {error}"
    _SEISMO_BENCH_RUN.record_manual_phase(
        _SEISMO_BENCH_PENDING_PHASE,
        elapsed,
        success=success,
        notes=notes,
    )
    _SEISMO_BENCH_PENDING_PHASE = None
    _SEISMO_BENCH_PENDING_T0 = None
    _seismo_bench_save_silently()

_seismo_bench_ip = get_ipython()
try:
    _seismo_bench_old_callback = getattr(
        _seismo_bench_ip, "_seismo_bench_post_run_cell_callback", None
    )
    if _seismo_bench_old_callback is not None:
        _seismo_bench_ip.events.unregister("post_run_cell", _seismo_bench_old_callback)
except Exception:
    pass
_seismo_bench_ip.events.register("post_run_cell", _seismo_bench_post_run_cell)
_seismo_bench_ip._seismo_bench_post_run_cell_callback = _seismo_bench_post_run_cell
print("Benchmark context ready:", _SEISMO_BENCH_CONTEXT)


<div style='background-image: url("../share/images/header.svg") ; padding: 0px ; background-size: cover ; border-radius: 5px ; height: 250px'>
    <div style="float: right ; margin: 50px ; padding: 20px ; background: rgba(255 , 255 , 255 , 0.7) ; width: 50% ; height: 150px">
        <div style="position: relative ; top: 50% ; transform: translatey(-50%)">
            <div style="font-size: xx-large ; font-weight: 900 ; color: rgba(0 , 0 , 0 , 0.8) ; line-height: 100%">Noise</div>
            <div style="font-size: large ; padding-top: 20px ; color: rgba(0 , 0 , 0 , 0.5)">Lab: Probabilistic Power Spectral Densities</div>
        </div>
    </div>
</div>


Seismo-Live: http://seismo-live.org

##### Authors:
* Tobias Megies ([@megies](https://github.com/megies))

---

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('setup/imports')


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use("bmh")
plt.rcParams['figure.figsize'] = 10, 6

 * read waveform data from file `data/GR.FUR..BHN.D.2015.361` (station `FUR`, [LMU geophysical observatory in Fürstenfeldbruck](https://www.geophysik.uni-muenchen.de/observatory/seismology))
 * read corresponding station metadata from file `data/station_FUR.stationxml`
 * print info on both waveforms and station metadata

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('data loading')


In [ ]:
from obspy import read, read_inventory

st = read("data/GR.FUR..BHN.D.2015.361")
inv = read_inventory("data/station_FUR.stationxml")

print(st)
print(inv)
inv.plot(projection="ortho");

 * compute probabilistic power spectral densities using `PPSD` class from obspy.signal, see http://docs.obspy.org/tutorial/code_snippets/probabilistic_power_spectral_density.html (but use the inventory you read from StationXML as metadata)
 * plot the processed `PPSD` (`plot()` method attached to `PPSD` object)

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('main compute')


In [ ]:
from obspy.signal import PPSD

tr = st[0]
ppsd = PPSD(stats=tr.stats, metadata=inv)

ppsd.add(tr)
ppsd.plot()

Since longer term stacks would need too much waveform data and take way too long to compute, we prepared one year continuous data preprocessed for a single channel of station `FUR` to play with..

 * load long term pre-computed PPSD from file `PPSD_FUR_HHN.npz` using `PPSD`'s `load_npz()` staticmethod (i.e. it is called directly from the class, not an instance object of the class)
 * plot the PPSD (default is full time-range, depending on how much data and spread is in the data, adjust `max_percentage` option of `plot()` option)  (might take a couple of minutes..!)
 * do a cumulative plot (which is good to judge non-exceedance percentage dB thresholds)

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('data loading')


In [ ]:
from obspy.signal import PPSD

ppsd = PPSD.load_npz("data/PPSD_FUR_HHN.npz",allow_pickle=True)

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('plotting/rendering')


In [ ]:
ppsd.plot(max_percentage=10)
ppsd.plot(cumulative=True)

 * do different stacks of the data using the [`calculate_histogram()` (see docs!)](http://docs.obspy.org/packages/autogen/obspy.signal.spectral_estimation.PPSD.calculate_histogram.html) method of `PPSD` and visualize them
 * compare differences in different frequency bands qualitatively (anthropogenic vs. "natural" noise)..
   * nighttime stack, daytime stack
 * advanced exercise: Use the `callback` option and use some crazy custom callback function in `calculate_histogram()`, e.g. stack together all data from birthdays in your family.. or all German holidays + Sundays in the time span.. or from dates of some bands' concerts on a tour.. etc.

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('main compute')


In [ ]:
ppsd.calculate_histogram(time_of_weekday=[(-1, 0, 2), (-1, 22, 24)])
ppsd.plot(max_percentage=10)
ppsd.calculate_histogram(time_of_weekday=[(-1, 8, 16)])
ppsd.plot(max_percentage=10)

 * do different stacks of the data using the [`calculate_histogram()` (see docs!)](http://docs.obspy.org/packages/autogen/obspy.signal.spectral_estimation.PPSD.calculate_histogram.html) method of `PPSD` and visualize them
 * compare differences in different frequency bands qualitatively (anthropogenic vs. "natural" noise)..
   * weekdays stack, weekend stack

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('main compute')


In [ ]:
ppsd.calculate_histogram(time_of_weekday=[(1, 0, 24), (2, 0, 24), (3, 0, 24), (4, 0, 24), (5, 0, 24)])
ppsd.plot(max_percentage=10)
ppsd.calculate_histogram(time_of_weekday=[(6, 0, 24), (7, 0, 24)])
ppsd.plot(max_percentage=10)

 * do different stacks of the data using the [`calculate_histogram()` (see docs!)](http://docs.obspy.org/packages/autogen/obspy.signal.spectral_estimation.PPSD.calculate_histogram.html) method of `PPSD` and visualize them
 * compare differences in different frequency bands qualitatively (anthropogenic vs. "natural" noise)..
   * seasonal stacks (e.g. northern hemisphere autumn vs. spring/summer, ...)

In [ ]:
# seismo-benchmark-control
_seismo_bench_start_phase('main compute')


In [ ]:
ppsd.calculate_histogram(month=[10, 11, 12, 1])
ppsd.plot(max_percentage=10)
ppsd.calculate_histogram(month=[4, 5, 6, 7])
ppsd.plot(max_percentage=10)

 * do different stacks of the data using the [`calculate_histogram()` (see docs!)](http://docs.obspy.org/packages/autogen/obspy.signal.spectral_estimation.PPSD.calculate_histogram.html) method of `PPSD` and visualize them
 * compare differences in different frequency bands qualitatively (anthropogenic vs. "natural" noise)..
   * stacks by specific month
   * maybe even combine several of above restrictions.. (e.g. only nighttime on weekends)

In [ ]:
# seismo-benchmark-control
_SEISMO_BENCH_RUN.record_manual_phase(
    "full notebook total",
    _seismo_bench_time.perf_counter() - _SEISMO_BENCH_TOTAL_T0,
)
_seismo_bench_csv_path, _seismo_bench_json_path = _SEISMO_BENCH_RUN.save()
print("Benchmark results written:", _seismo_bench_csv_path, _seismo_bench_json_path)
